# CrossFuse v5 -- Stage A2: FF++ Real-Video Extraction (for SBI pretraining)

> **STATUS: superseded by `A3-ffpp-full-extract.ipynb`, and the SBI stage it fed (`B0`) was cut.** Kept for the record; do not run for new work.

Extracts face crops from FaceForensics++ `original_sequences/youtube/c23`
(real videos only -- SBI needs no fakes). Attach the
[`xdxd003/ff-c23`](https://www.kaggle.com/datasets/xdxd003/ff-c23) Kaggle
mirror as input; no TUM download-form request needed.

Publishes `/kaggle/working/ffpp_real_crops_v5` -- publish it as a Kaggle
Dataset (e.g. `ffpp-real-crops-v5`) before running NB-B0.

Frames are cached at CROP_SIZE=256 with a generous margin (0.45, wider than
FakeAVCeleb's 0.30) since SBI's blending needs the surrounding jaw/cheek
region, not just a tight mouth-centric crop.


In [ ]:
!pip install -q --no-deps facenet-pytorch
import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))
print("Using crossfuse_v5 library from:", _lib_hits[0])

import os, glob
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import torch
from tqdm.auto import tqdm

from crossfuse_v5 import build_detector, extract_face_crops_contiguous, make_rng

N_DEVICES = max(1, torch.cuda.device_count())
DEVICE_IDS = [f"cuda:{i}" for i in range(N_DEVICES)] if torch.cuda.is_available() else ["cpu"]
print(f"Extraction devices: {DEVICE_IDS}")

OUT_DIR = "/kaggle/working/ffpp_real_crops_v5"
os.makedirs(OUT_DIR, exist_ok=True)

FRAMES_PER_VIDEO = 8     # SBI trains on independent FRAMES, not clips
CROP_SIZE = 256
FACE_MARGIN = 0.45


In [ ]:
videos = sorted(glob.glob("/kaggle/input/**/original_sequences/**/c23/**/*.mp4", recursive=True))
if not videos:
    # Some mirrors flatten the directory structure differently; fall back to
    # any mp4 whose path mentions "original" and does not mention a
    # manipulation method name.
    manip_tokens = ("deepfakes", "face2face", "faceswap", "neuraltextures", "manipulated")
    videos = [v for v in glob.glob("/kaggle/input/**/*.mp4", recursive=True)
              if "original" in v.lower() and not any(t in v.lower() for t in manip_tokens)]
print(f"Found {len(videos)} FF++ real videos")
assert videos, "No FF++ real videos found -- check the attached dataset's directory layout."


In [ ]:
# One detector per GPU, jobs round-robined across them via a thread pool --
# the same pattern NB-A uses. Extraction is I/O- and CPU-bound (video decode,
# MTCNN), so a thread per device overlaps them instead of running serially on
# a single GPU while the other sits idle.
def process_job(vi, vpath, device_str, mtcnn_local):
    key = f"ffppreal_{vi:04d}_{os.path.splitext(os.path.basename(vpath))[0]}"
    out_path = os.path.join(OUT_DIR, key + ".npy")
    if os.path.exists(out_path):
        return key, "cached"

    rng = make_rng(key)
    crops, valid, fps, start, ts = extract_face_crops_contiguous(
        vpath, mtcnn_local, rng=rng, n_frames=FRAMES_PER_VIDEO * 4,   # oversample, then subsample valid frames
        out_size=CROP_SIZE)
    if crops is None or int(valid.sum()) < FRAMES_PER_VIDEO:
        return key, "skipped"

    valid_idx = np.where(valid)[0]
    pick = rng.sample(list(valid_idx), FRAMES_PER_VIDEO) if len(valid_idx) >= FRAMES_PER_VIDEO \
           else list(valid_idx)
    np.save(out_path, crops[sorted(pick)])
    return key, "ok"


detectors = {d: build_detector(d) for d in DEVICE_IDS}
manifest = []
skipped = 0

with ThreadPoolExecutor(max_workers=len(DEVICE_IDS)) as pool:
    futures = [pool.submit(process_job, vi, vpath, DEVICE_IDS[vi % len(DEVICE_IDS)],
                           detectors[DEVICE_IDS[vi % len(DEVICE_IDS)]])
              for vi, vpath in enumerate(videos)]
    for fut in tqdm(as_completed(futures), total=len(futures),
                    desc=f"Extracting FF++ real crops ({len(DEVICE_IDS)} GPU(s))"):
        key, status = fut.result()
        if status in ("ok", "cached"):
            manifest.append(key)
        else:
            skipped += 1

print(f"\nExtracted {len(manifest)} videos ({skipped} skipped: no face / too short).")
print(f"Frames on disk: {len(manifest) * FRAMES_PER_VIDEO} (up to)")
with open(os.path.join(OUT_DIR, "manifest.txt"), "w") as f:
    f.write("\n".join(manifest))
print(f"Written to {OUT_DIR}. Save Version, publish as 'ffpp-real-crops-v5', attach to NB-B0.")
